# H3 — Faithful H0 Candidate Evidence Ranker

목표는 기존 H0 Selective-EB를 유지한 채 모든 26개 암종 후보의 evidence 분포를 shared pairwise ranker로 재정렬해 **큰 로컬 점프**를 확인하는 것입니다.

## 변경하지 않는 계약

- 기준 H0: `0.80 × Selective-EB LR + 0.20 × 자동 LGBM specialist`
- Selective-EB margin: 사전 고정 `0.05`
- outer CV: seed42 Stratified 5-fold / outer train 안쪽 inner 3-fold OOF
- 모든 event vocabulary, EB 가중치, ranking meta-feature, alpha 선택은 outer-train 안에서만 fit
- test.csv 미열람, train/test concat 금지, 고정 암종·유전자·exact mutation 규칙 금지
- correction strength는 inner OOF에서만 `{0.10, 0.20}` 중 하나를 고정 선택
- NaN/WT/blank는 이벤트가 아니며 `nan_as_mutation_count=0`을 검증

In [1]:
from pathlib import Path
import subprocess, sys
from tqdm.auto import tqdm

ROOT = Path('/Users/admin/Documents/FinalProject/OZ_fianl_hackaton')
EXP_DIR = ROOT / 'experiments' / 'gs' / 'notebooks' / 'exp_model_008'
RUNNER = EXP_DIR / 'common' / 'run_faithful_h0_candidate_ranker.py'
RESULT = EXP_DIR / 'result'
RUN_ID = 'exp-faithful-h0-candidate-ranker-01'
SEED = 42
RUN_EXPERIMENT = True
assert RUNNER.exists() and (ROOT / 'data/raw/train.csv').exists()
print({'runner': RUNNER, 'result': RESULT, 'seed': SEED, 'run': RUN_EXPERIMENT})

{'runner': PosixPath('/Users/admin/Documents/FinalProject/OZ_fianl_hackaton/experiments/gs/notebooks/exp_model_008/common/run_faithful_h0_candidate_ranker.py'), 'result': PosixPath('/Users/admin/Documents/FinalProject/OZ_fianl_hackaton/experiments/gs/notebooks/exp_model_008/result'), 'seed': 42, 'run': True}


/Users/admin/Documents/FinalProject/OZ_fianl_hackaton/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
if RUN_EXPERIMENT:
    process = subprocess.Popen(
        [sys.executable, str(RUNNER), '--seed', str(SEED), '--run-id', RUN_ID],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    tail = []
    for line in tqdm(process.stdout, desc='H3 faithful ranker folds', unit='line'):
        print(line, end='')
        tail = (tail + [line])[-120:]
    if process.wait():
        raise RuntimeError('H3 runner failed:\n' + ''.join(tail))
else:
    print('RUN_EXPERIMENT=False: 기존 결과만 읽습니다.')

H3 faithful ranker folds: 1line [00:04,  4.98s/line]

[H3] outer fold 1/5: inner OOF H0 Selective-EB meta-features


## 결과와 자동 판정

승격은 `+0.015`, 4/5 fold 상승, low-margin 보호, fold/class 편중 방지를 모두 만족해야 합니다.

In [ ]:
import json
import matplotlib.pyplot as plt
import pandas as pd

prefix = RESULT / f'{RUN_ID}_seed{SEED}'
summary = pd.read_csv(prefix.with_name(prefix.name + '_summary.csv'))
folds = pd.read_csv(prefix.with_name(prefix.name + '_fold_metrics.csv'))
classes = pd.read_csv(prefix.with_name(prefix.name + '_class_metrics.csv'))
topk = pd.read_csv(prefix.with_name(prefix.name + '_topk.csv'))
low = pd.read_csv(prefix.with_name(prefix.name + '_low_margin.csv'))
audit = json.loads(prefix.with_name(prefix.name + '_leakage_audit.json').read_text(encoding='utf-8'))

assert summary.leakage_check.all() and summary.nan_as_mutation_count.eq(0).all()
display(summary)
display(folds.pivot(index='fold', columns='variant', values='macro_f1'))
display(low)
audit['selection']

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
folds.pivot(index='fold', columns='variant', values='macro_f1').plot(marker='o', ax=axes[0], title='Fold Macro F1')
classes.sort_values('delta_f1').plot.barh(x='class', y='delta_f1', ax=axes[1], legend=False, title='Class F1 delta')
axes[1].axvline(0, color='black', linewidth=1)
topk.pivot(index='k', columns='variant', values='recall').plot.bar(ax=axes[2], title='Top-k recall')
plt.tight_layout(); plt.show()